In [1]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

from neuro_symbolic.pipeline import NeuroSymbolicPipeline
from symbolic.rule_engine import RuleEngine
from symbolic.quality_control_rules import get_quality_control_rules
from neural.llm_client import LLMClient

In [2]:
# Инициализация компонентов
engine = RuleEngine()
engine.add_rules(get_quality_control_rules())
print(f"Загружено правил: {len(engine.rules)}")
for r in engine.rules:
    print(f"  {r.rule_id}: {r.name} [{r.priority.name}]")

Загружено правил: 7
  QC_001: Размер вне допуска [CRITICAL]
  QC_003: Отсутствие компонента [CRITICAL]
  QC_007: Критическое превышение дефектов [CRITICAL]
  QC_002: Цветовой дефект [HIGH]
  QC_004: Поверхностный дефект [HIGH]
  QC_005: Смещение позиционирования [MEDIUM]
  QC_006: Низкая уверенность классификации [MEDIUM]


In [5]:
# Создание нейро-символьного пайплайна
pipeline = NeuroSymbolicPipeline(
    llm=LLMClient(),
    rule_engine=engine,
    neural_weight=0.6,
    symbolic_weight=0.4
)

In [6]:
# Тестовый сценарий: обнаружение дефектов на производственной линии
test_data = {
    'query': 'Деталь имеет размер 110 мм при допуске 105 мм, обнаружена царапина площадью 0.8 кв.мм, '
             'смещение позиционирования 2 мм, отсутствует один компонент.',
    'facts': {
        'dimension_mm': 110,
        'tolerance_upper': 105,
        'tolerance_lower': 95,
        'surface_defect_area': 0.8,
        'position_offset_mm': 2.0,
        'components_present': 2,
        'expected_components': 3,
        'defect_count': 3
    },
    'categories': ['годен', 'брак', 'предупреждение']
}

In [7]:
# Запуск пайплайна
result = pipeline.process(test_data)

print('='*60)
print('ФИНАЛЬНОЕ РЕШЕНИЕ:')
print(result['final_decision'])
print(f"\nИТОГОВАЯ УВЕРЕННОСТЬ: {result['confidence']:.1%}")
print(f"\nВРЕМЯ ВЫПОЛНЕНИЯ: {result['execution_time']} сек")
print('='*60)
print('ОБЪЯСНЕНИЕ РЕШЕНИЯ:')
print(result['explanation'])

ФИНАЛЬНОЕ РЕШЕНИЕ:
Нейронный вывод: Брак.
Символьный вывод: БРАК: Размер изделия не соответствует допускам, БРАК: Отсутствует один или несколько компонентов, ТРЕБУЕТСЯ ДОПОЛНИТЕЛЬНЫЙ КОНТРОЛЬ: Обнаружены царапины/сколы, ПРЕДУПРЕЖДЕНИЕ: Смещение при установке детали превышает 1 мм

ИТОГОВАЯ УВЕРЕННОСТЬ: 88.0%

ВРЕМЯ ВЫПОЛНЕНИЯ: 1.504 сек
ОБЪЯСНЕНИЕ РЕШЕНИЯ:
=== ОБЪЯСНЕНИЕ РЕШЕНИЯ ===

Нейронная компонента:
 • Классификация: брак
 • Уверенность: 80.0%

Символьная компонента:
 • Размер вне допуска: Геометрический размер выходит за границы допуска
 • Отсутствие компонента: Обнаружено отсутствие обязательного элемента сборки
 • Поверхностный дефект: Площадь поверхностного дефекта превышает 0.5 мм²
 • Смещение позиционирования: Позиционное смещение не соответствует спецификации

Итоговая уверенность: 88.0%
 • Вес нейронной компоненты: 60.0%
 • Вес символьной компоненты: 40.0%


In [8]:
# Второй сценарий: изделие без дефектов
test_data_ok = {
    'query': 'Деталь годна, все параметры в норме',
    'facts': {
        'dimension_mm': 100,
        'tolerance_upper': 105,
        'tolerance_lower': 95,
        'surface_defect_area': 0.0,
        'position_offset_mm': 0.2,
        'components_present': 3,
        'expected_components': 3,
        'defect_count': 0
    },
    'categories': ['годен', 'брак', 'предупреждение']
}

result_ok = pipeline.process(test_data_ok)
print(result_ok['final_decision'])
print(f"Уверенность: {result_ok['confidence']:.1%}")

Нейронный вывод: Деталь годна, все параметры находятся в пределах допустимых норм.
Уверенность: 48.0%
